In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10


In [10]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(), ##automatically scales & converts the images into pytorch tensors
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform= transform)
testset = CIFAR10(root="./data", train=False, download=True, transform= transform)



In [11]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

Build CNN

In [17]:
from torch.nn.modules.pooling import MaxPool2d
class CNN(nn.Module) :
  def __init__(self) :
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32,kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2), # kernel size = 2, stride=2

        nn.Conv2d(32, 64,kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),# kernel size = 2, stride=2

        nn.Conv2d(64, 128,kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2), # kernel size = 2, stride=2
     )

    #fully connected layers
    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128, 256),
        nn.ReLU(),

        nn.Linear(256, 10)
    )


  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1) #flattening
    x = self.fc_layers(x)

    return x



In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Training the CNN Model

In [23]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)  # no need .forward()
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs}, loss={epoch_training_loss/len(trainloader)}")

epoch=1/10, loss=0.14337348217940163
epoch=2/10, loss=0.11075189052378316
epoch=3/10, loss=0.09389078482459097
epoch=4/10, loss=0.09087022751524491
epoch=5/10, loss=0.08706333706586543
epoch=6/10, loss=0.07767572269722571
epoch=7/10, loss=0.07924598777908451
epoch=8/10, loss=0.07129167207299977
epoch=9/10, loss=0.06819785382676646
epoch=10/10, loss=0.07199841055057674


In [25]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
  for images, labels in testloader:
    outputs = model.forward(images)
    _, predicted = torch.max(outputs, 1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.96000000000001
